# 🎭 AI Avatar All-in-One - Làm Tất Cả Trên Google Colab

> **Bỏ video/ảnh vào → Colab lo hết → Ra kết quả live. Không cần GPU máy local!**

## ⚡ Cách Dùng (3 bước)

1. Upload video hoặc ảnh vào Google Drive folder `AI_Face_Data/input/`
2. **Bật GPU:** Runtime → Change runtime type → **T4 GPU**
3. Chạy từng cell: `Ctrl+Enter` hoặc Runtime → Run all

## 📋 Pipeline Tự Động

```
Video/Ảnh Input → Auto phát hiện mặt → Phân loại góc → Trích landmarks
→ Delaunay Face Warp → Tạo animation → Live demo webcam → Xuất video
```


## 1. Mount Drive & Cài Đặt


In [1]:
# ============================================================
# 1. MOUNT DRIVE + INSTALL + GIT SYNC
# ============================================================
from google.colab import drive
drive.mount('/content/drive')

# Sync code tu GitHub (bo comment neu muon dung)
# !rm -rf /content/train-ai
# !git clone https://github.com/YOUR_USERNAME/train-ai.git /content/train-ai
# %cd /content/train-ai

!pip install -q opencv-python face-recognition numpy scipy matplotlib
print("[OK] Thu vien san sang.")


Mounted at /content/drive
     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 100.1/100.1 MB 9.5 MB/s eta 0:00:00
  Preparing metadata (setup.py) ... done
[OK] Thu vien san sang.


In [2]:
print("[OK] Face Detection ready!")

# Default vars + paths (auto-train se ghi de)
OUTPUT_DIR = '/content/drive/MyDrive/AI_Face_Data/output'
best_frames = {'left': None, 'center': None, 'right': None}
best_info = {'left': (0,0,0,None), 'center': (0,0,0,None), 'right': (0,0,0,None)}
all_motion = []
missing = ['left','center','right']
session_output = f'{OUTPUT_DIR}/default'

GPU: GPU 0: Tesla T4 (UUID: GPU-c26f8a3b-aef8-bf4d-5943-e18e258e42b7)


## 3. AUTO-TRAIN: Load Input → Phát Hiện Mặt → Phân Loại Góc


In [4]:
# ============================================================
# 2. AUTO-TRAIN: Scan + Detect + Classify
# ============================================================
# Scan all files (case-insensitive)
all_files = []
for ext in ['*.mp4','*.MP4','*.avi','*.AVI','*.mov','*.MOV',
            '*.jpg','*.JPG','*.jpeg','*.JPEG','*.png','*.PNG']:
    all_files.extend(Path(INPUT_DIR).glob(ext))
for d in Path(INPUT_DIR).iterdir():
    if d.is_dir():
        for ext in ['*.mp4','*.MP4','*.avi','*.AVI','*.mov','*.MOV',
                    '*.jpg','*.JPG','*.jpeg','*.JPEG','*.png','*.PNG']:
            all_files.extend(d.glob(ext))

new_files = [f for f in all_files if f.name not in trained_log]
done_files = [f for f in all_files if f.name in trained_log]
print(f"Files: {len(all_files)} total | {len(new_files)} new | {len(done_files)} done")
for f in new_files: print(f"  [NEW] {f.name}")

if len(new_files) == 0:
    print("[OK] No new files. Skipping training.")
    SESSION_ID = "no_new_files"
    session_output = f'{OUTPUT_DIR}/{SESSION_ID}'
    missing = ['left','center','right']
    best_frames = {'left':None,'center':None,'right':None}
    best_info = {'left':(0,0,0,None),'center':(0,0,0,None),'right':(0,0,0,None)}
    all_motion = []
else:
    SESSION_ID = datetime.now().strftime("%Y%m%d_%H%M%S")
    session_output = f'{OUTPUT_DIR}/{SESSION_ID}'
    os.makedirs(session_output, exist_ok=True)
    print(f"Output: {session_output}")

    best_frames = {'left':None,'center':None,'right':None}
    best_info = {'left':(0,0,0,None),'center':(0,0,0,None),'right':(0,0,0,None)}
    all_motion = []

    def process_frame(img, idx, ts=0):
        lm, yaw, _ = get_landmarks_and_pose(img)
        if lm is None: return
        q = cv2.Laplacian(cv2.cvtColor(img, cv2.COLOR_RGB2GRAY), cv2.CV_64F).var()
        angle = 'left' if yaw<-12 else ('right' if yaw>12 else 'center')
        all_motion.append((ts, yaw))
        _, _, bq, _ = best_info[angle]
        if bq is None or q > bq:
            best_frames[angle] = img.copy()
            best_info[angle] = (yaw, q, q, lm)

    for f in new_files:
        fn = str(f)
        if f.suffix.lower() in ['.mp4','.avi','.mov']:
            cap = cv2.VideoCapture(fn)
            if not cap.isOpened():
                print(f"[SKIP] Cannot open: {f.name}")
                continue
            fps = cap.get(cv2.CAP_PROP_FPS) or 30
            idx = 0
            while True:
                ret, frame = cap.read()
                if not ret: break
                if idx % 3 == 0:
                    process_frame(cv2.cvtColor(frame, cv2.COLOR_BGR2RGB), idx, idx/fps)
                idx += 1
            cap.release()
            print(f"[VIDEO] {f.name}: {idx} frames, {len([x for x in all_motion])} face detections")
        else:
            img = cv2.imread(fn)
            if img is None:
                print(f"[SKIP] Cannot read: {f.name}")
                continue
            process_frame(cv2.cvtColor(img, cv2.COLOR_BGR2RGB), 0, 0)
            print(f"[IMAGE] {f.name}")

    print(f"\nMotion points: {len(all_motion)}")
    missing = []
    for a in ['left','center','right']:
        if best_frames[a] is not None:
            y, q, _, _ = best_info[a]
            print(f"  {a}: yaw={y:.0f} quality={q:.0f}")
        else:
            print(f"  {a}: NOT FOUND")
            missing.append(a)

    if len(missing) < 3:
        fig, axes = plt.subplots(1, 3, figsize=(12,4))
        for i, a in enumerate(['left','center','right']):
            if best_frames[a] is not None:
                axes[i].imshow(best_frames[a])
                axes[i].set_title(f'{a} yaw={best_info[a][0]:.0f}')
            else:
                axes[i].text(0.5,0.5,'MISSING', ha='center', va='center', fontsize=20)
            axes[i].axis('off')
        plt.tight_layout(); plt.show()

    for f in new_files:
        trained_log[f.name] = {'date': datetime.now().strftime("%Y-%m-%d %H:%M:%S"), 'output_subfolder': SESSION_ID}
    with open(LOG_FILE, 'w') as fl: json.dump(trained_log, fl, indent=2)
    print(f"[LOG] Saved {len(trained_log)} files to log")


SyntaxError: incomplete input (1781229187.py, line 9)

## 4. Face Warp Engine: Xoay Mặt Mượt Bằng Delaunay


## 5. Tạo Video Animation Tự Nhiên


In [ ]:
# ============================================================
# 5. CREATE ANIMATION VIDEO
# ============================================================
if missing:
    print("SKIP: Thieu du lieu goc")
else:
    print("Creating animation: left -> center -> right -> center...")

    # Motion curve: if we have motion data from video, use it; else generate smooth curve
    if len(all_motion) > 10:
        # Use real motion from input video
        timestamps, yaws = zip(*all_motion)
        print(f"Using real motion: {len(yaws)} points")
    else:
        # Generate smooth curve
        frames_per_segment = 60
        yaw_range_left = list(np.linspace(yaw_left, yaw_center, frames_per_segment))
        yaw_range_right = list(np.linspace(yaw_center, yaw_right, frames_per_segment))
        yaws = yaw_range_left + yaw_range_right[1:] + list(reversed(yaw_range_left))[1:]
        print(f"Using generated motion: {len(yaws)} frames")

    # Create video - luu vao session folder
    out_path = f'{session_output}/avatar_animation.mp4'
    fourcc = cv2.VideoWriter_fourcc(*'mp4v')
    out = cv2.VideoWriter(out_path, fourcc, 30, (w, h))

    frame_count = 0
    for target_yaw in tqdm(yaws[:300], desc="Rendering"):  # Limit to 300 frames
        rendered = rotate_to(target_yaw)

        # Add UI
        rendered_ui = rendered.copy()
        cv2.putText(rendered_ui, f"Yaw: {target_yaw:+.0f} deg",
                    (10,30), cv2.FONT_HERSHEY_SIMPLEX, 0.6, (0,255,0), 2)

        out.write(cv2.cvtColor(rendered_ui, cv2.COLOR_RGB2BGR))
        frame_count += 1

    out.release()
    print(f"\n[OK] Video saved: {out_path}")
    print(f"     Frames: {frame_count}, Size: {w}x{h}")

    # Play in notebook
    from IPython.display import Video
    Video(out_path, width=400)


## 6. Live Demo: Chụp Webcam → Nhận Diện → So Sánh Với Dataset


In [ ]:
# ============================================================
# 6. LIVE DEMO: Webcam Capture + Angle Recognition
# ============================================================
from IPython.display import display, Javascript, Image as IPImage
from google.colab.output import eval_js
from base64 import b64decode
import PIL.Image
import io

def take_photo():
    js = Javascript('''
    async function takePhoto() {
        const div = document.createElement('div');
        const btn = document.createElement('button');
        btn.textContent = 'CHUP ANH';
        btn.style.cssText = 'padding:10px 30px; font-size:18px; margin:10px; cursor:pointer; background:#4CAF50; color:white; border:none; border-radius:5px';
        div.appendChild(btn);
        const video = document.createElement('video');
        video.style.display = 'block';
        const stream = await navigator.mediaDevices.getUserMedia({video: true});
        document.body.appendChild(div);
        div.appendChild(video);
        video.srcObject = stream;
        await video.play();
        google.colab.output.setIframeHeight(document.documentElement.scrollHeight, true);
        await new Promise((resolve) => btn.onclick = resolve);
        const canvas = document.createElement('canvas');
        canvas.width = video.videoWidth;
        canvas.height = video.videoHeight;
        canvas.getContext('2d').drawImage(video, 0, 0);
        stream.getVideoTracks()[0].stop();
        div.remove();
        return canvas.toDataURL('image/jpeg', 0.8);
    }
    ''')
    display(js)
    data = eval_js('takePhoto()')
    img_bytes = b64decode(data.split(',')[1])
    return cv2.cvtColor(cv2.imdecode(np.frombuffer(img_bytes, np.uint8), 1), cv2.COLOR_BGR2RGB)

print("Nhan nut 'CHUP ANH' ben duoi de test...")
photo = take_photo()

if photo is not None:
    lm, yaw, _ = get_landmarks_and_pose(photo)
    if lm is not None:
        # Draw landmarks
        display_img = photo.copy()
        for pt in lm.astype(np.int32):
            cv2.circle(display_img, tuple(pt), 2, (0,255,0), -1)

        # Classify
        if yaw < -12: label = f"QUAY TRAI ({yaw:.0f} deg)"
        elif yaw > 12: label = f"QUAY PHAI ({yaw:.0f} deg)"
        else: label = f"NHIN THANG ({yaw:.0f} deg)"

        cv2.putText(display_img, label, (10,40), cv2.FONT_HERSHEY_SIMPLEX, 0.9, (0,255,0), 2)

        # If we have the warp engine, show the corresponding avatar angle
        if not missing and 'rotate_to' in dir():
            avatar = rotate_to(yaw)
            avatar = cv2.resize(avatar, (photo.shape[1]//2, photo.shape[0]//2))

        plt.figure(figsize=(10,5))
        plt.subplot(1,2,1)
        plt.imshow(display_img)
        plt.title('Webcam')
        plt.axis('off')
        if not missing:
            plt.subplot(1,2,2)
            plt.imshow(avatar)
            plt.title(f'AI Avatar (yaw={yaw:.0f})')
            plt.axis('off')
        plt.tight_layout()
        plt.show()
    else:
        print("Khong tim thay khuon mat!")


## 7. Xuất Model & Download


In [ ]:
# ============================================================
# 7. EXPORT MODEL + SAVE ALL
# ============================================================
if not missing:
    # Save face data for local use
    face_data = {
        'images': {
            'left': img_left,
            'center': img_center,
            'right': img_right,
        },
        'landmarks': {
            'left': lm_left,
            'center': lm_center,
            'right': lm_right,
        },
        'yaws': {
            'left': yaw_left,
            'center': yaw_center,
            'right': yaw_right,
        },
        'motion': all_motion,
    }

    # Save best frames vao session folder
    for angle in ['left','center','right']:
        if best_frames[angle] is not None:
            cv2.imwrite(f'{session_output}/best_{angle}.jpg',
                        cv2.cvtColor(best_frames[angle], cv2.COLOR_RGB2BGR))

    # Save full pickle
    with open(f'{session_output}/face_data.pkl', 'wb') as f:
        pickle.dump(face_data, f)

    print(f"[OK] Saved to {session_output}/")
    for f in os.listdir(session_output):
        size = os.path.getsize(f'{session_output}/{f}') / 1024
        print(f"  {f} ({size:.1f} KB)")
else:
    print(f"SKIP: Thieu du lieu de xuat (missing: {missing})")


## 8. Tổng Kết

✅ **Pipeline Hoàn Thành!**

📥 **Kết quả lưu tại:** `AI_Face_Data/output/[SESSION_ID]/`

- `avatar_animation.mp4` — Video animation
- `best_left/center/right.jpg` — Ảnh tốt nhất mỗi góc
- `face_data.pkl` — Model cho máy local

📋 **Nhật ký:** `AI_Face_Data/trained_log.json` — tự ghi file nào đã train

🔁 **Lần sau có video mới:**

1. Upload vào `AI_Face_Data/input/`
2. Restart & Run All → tự bỏ qua video cũ, chỉ train video mới
3. Kết quả ra folder riêng: `output/20260727_143000/`

🗑️ **Muốn train lại video cũ:** Xóa tên file trong `trained_log.json` hoặc đổi tên video
